# Illustration Scrapping Studio — Jupyter
Runs the complete app on this server. Your browser only controls and reviews it. No training integration.

## Order of cells
1. **Configuration** — set `APP` to the project directory (the one containing `studio.py`).
2. **Setup/update** — stops the owned app, checks for surviving processes, updates a Git checkout, installs pinned dependencies, and refreshes the UI. Rerun this after every update.
3. **Start once** — launches the app and prints the link to open.
4. **Stop** — graceful shutdown, only when you want it.
5. **Diagnose** — explains a start failure, such as an occupied port.

## The UI comes from GitHub
`frontend/dist` (the built UI) is git-ignored, so `git pull` only updates the source. With Node installed, setup rebuilds the UI itself. Without Node, setup downloads the UI that GitHub built for exactly the frontend source you just pulled (the "UI build" workflow publishes one for every version), waiting a few minutes if that build is still running, and skips the download when the served UI already matches. Offline servers can set `DOWNLOAD_UI = False` in the setup cell and upload `frontend-ui.zip` to `UI_ZIP` instead.

Keep your own copy of this notebook outside the app folder (File → Save Notebook As…), so editing your paths never blocks `git pull`.

Do not upload your laptop's `.venv`, credentials, or library as part of the source code.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, importlib.util
APP = Path('/home/jovyan/Illustration-Scrapping-Studio').resolve()  # EDIT to the actual project directory
LIBRARY = Path('/home/jovyan/illustration-library').resolve()  # keep on persistent storage
UI_ZIP = Path('/home/jovyan/frontend-ui.zip').resolve()  # only used with DOWNLOAD_UI = False (offline servers)
DIST = APP / 'frontend' / 'dist'  # the exact folder the backend serves
PORT = 8000
assert (APP / 'studio.py').is_file(), 'Set APP to the directory containing studio.py'
assert sys.version_info >= (3, 12), 'Python 3.12+ required'
assert os.environ.get('JUPYTERHUB_SERVICE_PREFIX'), 'Run inside your authenticated JupyterHub session'
BASE_PATH = os.environ['JUPYTERHUB_SERVICE_PREFIX'].rstrip('/') + f'/proxy/{PORT}'
print('Persistent library:', LIBRARY)
print('Served UI folder:  ', DIST)
print('Browser path:', BASE_PATH + '/')
UPDATE_SOURCE = True  # False for offline setup; source archives do not run git pull
spec = importlib.util.spec_from_file_location('jupyter_support', APP / 'tools/jupyter_support.py')
studio_support = importlib.util.module_from_spec(spec)
spec.loader.exec_module(studio_support)


In [ ]:
# Setup/update. Finish jobs first. A failed graceful stop aborts this cell.
DOWNLOAD_UI = True  # without Node: download the UI GitHub built for this version; False installs the uploaded UI_ZIP
studio_support.stop_app(globals().get('studio_process'))
studio_support.assert_stopped(APP, PORT)
has_npm = bool(shutil.which('npm'))
is_git = (APP / '.git').exists()
if UPDATE_SOURCE and is_git:
    pulled = subprocess.run(['git', '-C', str(APP), 'pull', '--ff-only', 'origin', 'main'], capture_output=True, text=True)
    print(pulled.stdout.strip())
    if pulled.returncode:
        if 'would be overwritten' in pulled.stderr and '.ipynb' in pulled.stderr:
            raise RuntimeError('git pull stopped because the notebook inside the app folder has your edits. Save your copy elsewhere '
                               '(File > Save Notebook As..., outside the app folder), then run in a terminal:\n'
                               f'  git -C "{APP}" checkout -- "Jupyter Studio.ipynb"\nand rerun this cell from your copy.')
        raise RuntimeError(f'git pull failed:\n{pulled.stderr}')
    spec.loader.exec_module(studio_support)  # use the helper that came with this update
elif UPDATE_SOURCE:
    print('Source archive detected: skipping git pull. Replace source manually for updates.')
# On a no-Node install the UI must be in place BEFORE setup --skip-ui-build.
if not has_npm:
    installed = studio_support.fetch_ui(APP) if DOWNLOAD_UI and is_git else studio_support.refresh_ui(APP, UI_ZIP)
    print('Installed UI:', installed)
    print('Assets:', sorted(p.name for p in (installed / 'assets').glob('*')))
command = [sys.executable, str(APP / 'studio.py'), 'setup']
if not has_npm:
    command.append('--skip-ui-build')
subprocess.run(command, cwd=APP, check=True)
print('Setup/update complete. Run the Start once cell next. Refresh this notebook from the repository when its instructions change.')


In [ ]:
# Start once. A separate process continues when this browser disconnects.
# Server/container shutdown still stops it; restart explicitly after that.
import time, urllib.request, json
from IPython.display import display, HTML
from html import escape
if globals().get('studio_process') is not None and studio_process.poll() is None:
    raise RuntimeError('This notebook already started a running app')
studio_support.assert_stopped(APP, PORT)
LIBRARY.mkdir(parents=True, exist_ok=True)
launch_log = LIBRARY / 'launcher.log'
environment = dict(os.environ)
# Prevent inherited paths from silently targeting another library/database.
for key in ('ARTIST_DB_PATH', 'ARTIST_LIBRARY_PATH', 'ARTIST_LOG_PATH', 'ARTIST_PROVIDER_SETTINGS_PATH'):
    environment.pop(key, None)
with launch_log.open('ab') as output:
    studio_process = subprocess.Popen([str(APP / '.venv/bin/python'), str(APP / 'studio.py'), 'run', '--library', str(LIBRARY), '--database', str(LIBRARY / 'index.db'), '--port', str(PORT), '--base-path', BASE_PATH, '--no-browser'], cwd=APP, env=environment, stdout=output, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
for attempt in range(120):
    if studio_process.poll() is not None:
        tail = launch_log.read_text(errors='replace')[-2000:] if launch_log.is_file() else '(no launcher.log)'
        raise RuntimeError(f'App exited. The reason is the last line of launcher.log:\n\n{tail}')
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/health', timeout=1) as response:
            health = json.load(response)
        if health['library_path'] != str(LIBRARY):
            raise RuntimeError('A different app already owns this port. Choose another PORT and rerun configuration.')
        time.sleep(1)
        if studio_process.poll() is not None:
            tail = launch_log.read_text(errors='replace')[-2000:] if launch_log.is_file() else '(no launcher.log)'
            raise RuntimeError(f'App exited. The reason is the last line of launcher.log:\n\n{tail}')
        display(HTML(f'<a href="{escape(BASE_PATH + "/", quote=True)}" target="_blank" rel="noopener">Open Illustration Scrapping Studio</a>'))
        break
    except OSError:
        time.sleep(0.5)
else:
    raise RuntimeError('Startup not ready. Read launcher.log; the process may still be starting.')

In [ ]:
# Stops only the app owned by this kernel. Safe to run repeatedly.
STOP_APP = False  # Set True when you want to stop the app.
if STOP_APP:
    studio_support.stop_app(globals().get('studio_process'))
    studio_process = None
    print('Owned app stopped, or was already stopped. After a kernel restart, use Diagnose to locate older processes.')
else:
    print('App left running. Set STOP_APP=True to request shutdown.')


## Reconnect and maintenance
Reopen the printed proxy link to reconnect. If the kernel restarts, do not blindly launch a second copy: the backend may still be running. Library locks and port checks reject duplicate starts. Use your terminal/process manager to stop the known app if the notebook has lost its process handle.

Provider credentials entered/uploaded through this UI are stored on the server. Use the authenticated Jupyter link; do not put this port behind a public anonymous tunnel. Back up privately with `studio.py backup` after stopping the backend; backups include credentials. See `docs/JUPYTER.md` for setup, updates and troubleshooting.


In [ ]:
# Read-only diagnostics; never kill processes based on a broad name pattern.
try:
    studio_support.assert_stopped(APP, PORT)
    print('No running app found for this checkout; configured port is free.')
except RuntimeError as error:
    print(error)
pids = studio_support.app_processes(APP)
for pid in pids:
    print(f'App PID {pid}: inspect in a terminal with: ps -p {pid} -o pid,args')
    print(f'After verifying this is your app, request graceful stop with: kill -TERM {pid}')
if not pids:
    print(f'If the port is occupied by another program, identify it with: ss -ltnp sport = :{PORT}')
log = LIBRARY / 'launcher.log'
if log.is_file():
    print(log.read_text(errors='replace')[-2000:])
